# People by country

Answer the question **"who has a documented connection (stay, arrival, departure,
vows, ordination) with a place in country X?"** — for example *all the people
whose location attributes point to a place in **China***.

### What this notebook does

1. Reads every location-bearing attribute of a person from the database
   (`estadia`, `chegada`, `partida`, `jesuita-entrada`, `jesuita-votos-local`,
   `jesuita-ordenacao-padre`, `morte`, `residencia`, `baptizado`,
   `jesuita-fundador`, `trajeto` and their `-x` variants). An **inventory of every
   attribute type** with sample values and a `counted_as_place` column follows the
   load, so the classification can be checked.
2. Extracts the Wikidata id (Q-id) linked to each place value.
3. Resolves each Q-id to its **country**, using the cache
   `../inferences/wikidata-references/locations_wikidata_info.xlsx`
   (maintained by [wikidata-linked-data](./wikidata-linked-data.ipynb)), with an
   optional live lookup on the Wikidata API for Q-ids that are not in the cache.
4. Groups historical states with the modern country (`COUNTRY_GROUPS`), so that
   *Ming dynasty*, *Qing dynasty* and *Republic of China* are counted as **China**.
5. Lets you pick a country and prints **who was where**, plus flat tables and an
   Excel export under `../inferences/people-by-country/`.

### How it differs from the other notebooks

| notebook | what it gives |
|---|---|
| [location-analysis-new](./location-analysis-new.ipynb) | every place attribute with inferred dates and Wikidata labels, and *aggregate* counts per country |
| [who-was-where](./who-was-where.ipynb) | people at a **hard-coded list of Q-ids** |
| [residences](./residences.ipynb) | people in Dehergne's residence lists of 1644 and 1701 (China only) |
| **this notebook** | all people at **any place of a chosen country**, from the biographical attributes |

> **Attribute names.** In this transcription "local de votos" is
> `jesuita-votos-local` and "local de ordenação" is `jesuita-ordenacao-padre`.
> `jesuita-entrada` is the place of the **novitiate**, where the person lived for
> the years of his probation, and `morte` is where he died — both are places where
> he was present, so they count alongside `estadia`.
>
> A trailing **`-x`** marks information that was **not in the original entry** and
> was added from another source. It is the same information, so if an attribute
> counts, its `-x` variant counts too (`estadia-x`, `partida-x`, …).
>
> `nascimento` (birth) is the one place attribute left out by default; add it in
> `ALL_PLACE_ATTRIBUTES` below.

## Setup

In [ ]:
from timelink.notebooks import TimelinkNotebook

tlnb = TimelinkNotebook(kleio_image="kleio-server", kleio_version="12.9.592")
db = tlnb.db
tlnb.print_info()

## Which attributes count as a location

In [ ]:
# Attribute types whose VALUES are places, and where the person is documented as
# having been. Established by listing every attribute type in the database with a
# sample of its values (see the inventory in the next-but-one cell).
BASE_PLACE_ATTRIBUTES = [
    "estadia",                  # stay / residence
    "chegada",                  # arrival
    "partida",                  # departure
    "jesuita-entrada",          # entry into the novitiate: he lived there
    "jesuita-votos-local",      # place of vows       ("local de votos")
    "jesuita-ordenacao-padre",  # place of ordination ("local de ordenação")
    "morte",                    # place of death: he was there
    "residencia",               # residence (mostly of relatives recorded in an entry)
    "baptizado",                # place of baptism: he was there
    "jesuita-fundador",         # place where the Society was founded (Paris, 1534)
    "trajeto",                  # place/route of a journey (e.g. the Volga)
]

# A trailing "-x" marks information that was NOT in the original entry but was
# added from another source. It is the same information, so if an attribute
# counts, its "-x" variant counts too. Variants that do not occur simply match
# nothing (here: estadia-x and partida-x).
def with_uncertain(attributes):
    return attributes + [f"{the_type}-x" for the_type in attributes]


PLACE_ATTRIBUTES = with_uncertain(BASE_PLACE_ATTRIBUTES)

# Left out by default: `nascimento` is as much a place of origin as of presence.
ALL_PLACE_ATTRIBUTES = with_uncertain(BASE_PLACE_ATTRIBUTES + ["nascimento"])

PLACE_ATTRIBUTES

## Load the location attributes from the database

Attributes whose place value is `?` are dated but have no identified place: they
are dropped here, because they cannot be attributed to anywhere. The table below
shows how many that is, per attribute type.

In [ ]:
import pandas as pd
from timelink.pandas import entities_with_attribute

pd.set_option("display.max_rows", 200)

places = entities_with_attribute(
    entity_type="person",
    show_elements=["name", "groupname", "the_line", "the_order", "extra_info"],
    the_type=PLACE_ATTRIBUTES,
    column_name="place",
    db=db,
)

# the index holds the person id, e.g. deh-alexandre-cazalez
places["person_id"] = places.index
places = places.sort_values(["person_id", "place.line"])

# A place value of '?' (or an empty one) is a placeholder: the attribute is
# dated, but the transcription did not identify the place. Such a row carries no
# place and cannot be attributed to a country, so it is dropped.
placeholder = places["place"].isna() | (places["place"].astype(str).str.strip() == "?")
if placeholder.any():
    print(f"{placeholder.sum()} of {len(places)} attributes have no place ('?'), dropped:")
    print(places.loc[placeholder, "place.type"].value_counts().to_string())
    print()
places = places[~placeholder].copy()

print(f"{len(places)} location attributes of {places['person_id'].nunique()} people")
places[["name", "groupname", "place", "place.type", "place.date"]].head()

### Inventory of attribute types

Every attribute type in the database, with the number of rows and a sample of its
values, and whether this notebook counts it as a place. This is how the list
above was established — and how to check it when the transcription changes.

`nacionalidade` is a country, but of *origin* rather than of presence;
`embarque` holds ship names; the rest are dates, statuses, offices or names.

In [ ]:
from sqlalchemy import select

nattributes = db.get_view("nattributes")

with db.session() as session:
    rows = session.execute(
        select(nattributes.c.id, nattributes.c.the_type, nattributes.c.the_value)
        .where(nattributes.c.the_type.notlike("%@%"))  # skip the linked-data aspects
    ).all()

inventory_all = pd.DataFrame(rows, columns=["id", "the_type", "the_value"])
inventory_all["the_value"] = inventory_all["the_value"].astype(str).str.strip()
# '?' means the place was not identified: there is nothing to judge the type by
no_place = inventory_all["the_value"].isin(["", "?", "None"])
usable = inventory_all[~no_place]

inventory = (
    usable.groupby("the_type")
    .agg(
        rows=("the_value", "size"),
        people=("id", "nunique"),
        examples=("the_value", lambda values: "  |  ".join(values.drop_duplicates().head(3))),
    )
    # keep every type, even one whose values are all '?'
    .reindex(sorted(inventory_all["the_type"].unique()))
)
inventory["no_place"] = (
    no_place.groupby(inventory_all["the_type"]).sum().reindex(inventory.index).fillna(0).astype(int)
)
inventory[["rows", "people"]] = inventory[["rows", "people"]].fillna(0).astype(int)
inventory["examples"] = inventory["examples"].fillna("")
inventory["counted_as_place"] = [the_type in set(PLACE_ATTRIBUTES) for the_type in inventory.index]
inventory["left_out_by_default"] = [
    the_type in set(ALL_PLACE_ATTRIBUTES) and the_type not in set(PLACE_ATTRIBUTES)
    for the_type in inventory.index
]

pd.set_option("display.max_rows", 60)
pd.set_option("display.max_colwidth", 54)
inventory.sort_values("rows", ascending=False)

## Extract the Wikidata id of each place

In [ ]:
import json

from dehergne_util import extract_wikidata_id


def as_dict(value):
    """``extra_info`` comes back as a dict, but may be a JSON string.

    Normalise it so that :func:`extract_wikidata_id` can read the comment.
    """
    if isinstance(value, dict):
        return value
    if isinstance(value, str) and value.strip().startswith("{"):
        try:
            return json.loads(value)
        except ValueError:
            return {}
    return {}


places["place.extra_info"] = places["place.extra_info"].apply(as_dict)
places["wikidata_id"] = places["place.extra_info"].apply(
    extract_wikidata_id, if_missing=pd.NA
)

print(f"location attributes with a Wikidata id: "
      f"{places['wikidata_id'].notna().sum()} / {len(places)}")
places["place.type"].value_counts()

## Attach the country of each place

`locations_wikidata_info.xlsx` is the Wikidata cache built by
[wikidata-linked-data](./wikidata-linked-data.ipynb): one row per Q-id with
labels, coordinates and the `P17` (*country*) statement.

In [ ]:
from dehergne_util import locations_wikidata_info_file

wd_info = pd.read_excel(locations_wikidata_info_file, dtype=str)
wd_info = (
    wd_info.drop_duplicates(subset=["wikidata_id"], keep="first")
    .set_index("wikidata_id")
)

WD_COUNTRY_COLUMNS = [
    "country_id",
    "country_label",
    "english_label",
    "chinese_label",
    "portuguese_label",
    "administrative_entity_id",
    "administrative_entity_label_en",
    "latitude",
    "longitude",
]

places = places.join(wd_info[WD_COUNTRY_COLUMNS], on="wikidata_id", how="left")
places[["name", "place", "place.type", "wikidata_id", "country_label", "english_label"]].head()

## Nested places: a city and the province around it

Many people are documented both at a city and at the province that contains it —
`Cantão` (Guangzhou) in one entry, `Kwangtung` (Guangdong) in another. Counted
naively the same person appears twice, at two administrative levels.

`most_specific` marks the **deepest place** of each person: a row is `False`
when the same person is also documented in a place that lies inside it, using
the Wikidata `P131` (*located in*) chain in the cache. **No row is deleted** —
the flag only decides what the aggregations count, and it travels into the
export. Set `KEEP_ONLY_MOST_SPECIFIC = False` to count every row again.

Places with no Wikidata id cannot be placed in the hierarchy, so they are always
kept — a person may therefore still appear both under an unlinked spelling and
under the linked place.

In [ ]:
# Wikidata P131 ("located in the administrative territorial entity"), as cached
# in locations_wikidata_info.xlsx: each place -> the place that contains it.
PARENT_OF = {
    qid: parent
    for qid, parent in wd_info["administrative_entity_id"].items()
    if isinstance(parent, str) and parent
}


def ancestors(qid):
    """Every ancestor of a place, walking the P131 chain through the cache."""
    out, seen = set(), {qid}
    parent = PARENT_OF.get(qid)
    while parent and parent not in seen:
        out.add(parent)
        seen.add(parent)
        parent = PARENT_OF.get(parent)
    return out


ANCESTORS = {qid: ancestors(qid) for qid in places["wikidata_id"].dropna().unique()}

places_by_person = places.groupby("person_id")["wikidata_id"].apply(
    lambda values: {value for value in values if isinstance(value, str)}
)


def is_most_specific(row):
    """False when a more specific place of the same person contains this one."""
    qid = row["wikidata_id"]
    if not isinstance(qid, str):
        return True  # no Wikidata id: nothing to compare with
    return not any(
        qid in ANCESTORS.get(other, ()) for other in places_by_person[row["person_id"]]
    )


places["most_specific"] = places.apply(is_most_specific, axis=1)

# Count each person only at the most specific place documented for them.
# `counted_places` is built further down, once the country column exists.
KEEP_ONLY_MOST_SPECIFIC = True

shadowed = places[~places["most_specific"]]
print(
    f"{len(shadowed)} of {len(places)} location attributes are in a place that "
    f"contains another place of the same person"
)
print(f"{shadowed['person_id'].nunique()} people are affected")

### What the rule drops

Audit table: the places that lose people because those people are documented
somewhere more specific inside them.

In [ ]:
shadowed_summary = (
    shadowed.groupby(["wikidata_id", "english_label"], dropna=False)
    .agg(people=("person_id", "nunique"), attributes=("person_id", "size"))
    .sort_values("people", ascending=False)
)
shadowed_summary.head(15)

## How complete is the country information?

In [ ]:
total = len(places)
with_qid = places["wikidata_id"].notna().sum()
with_country = places["country_label"].notna().sum()

print(f"location attributes      : {total}")
print(f"  ... with a Wikidata id : {with_qid} ({with_qid / total:.0%})")
print(f"  ... with a country     : {with_country} ({with_country / total:.0%})")

no_country = sorted(
    set(
        places.loc[
            places["wikidata_id"].notna() & places["country_label"].isna(), "wikidata_id"
        ]
    )
)
print(f"\n{len(no_country)} Wikidata id(s) used here have no country in the cache:")
for qid in no_country:
    example = places.loc[places["wikidata_id"] == qid, "place"].iloc[0]
    print(f"  {qid}  {example}")

### Optional: fetch the missing countries from Wikidata

The next cell looks up, **only for the Q-ids listed above**, the `P17` (*country*)
statement on Wikidata — falling back to the country of the first `P131`
(*located in*) entity, which is what historical and administrative places need.
Results are cached in `../inferences/wikidata-references/wd_country_cache.json`
so the lookup happens once.

Set `FETCH_MISSING_COUNTRIES = False` to work offline.

In [ ]:
FETCH_MISSING_COUNTRIES = True

COUNTRY_CACHE_FILE = "../inferences/wikidata-references/wd_country_cache.json"


def fetch_country_for_qids(qids, cache_file=COUNTRY_CACHE_FILE):
    """Country id and label of each Wikidata id, using P17 and a JSON cache.

    Only Q-ids not yet cached are fetched, in batches of 50 (the Wikidata API
    limit). Returns {qid: {"country_id": ..., "country_label": ...}}.
    """
    import time
    from pathlib import Path
    from urllib.request import Request, urlopen

    def get(ids, props):
        """One API call, retried; returns {} when Wikidata cannot be reached.

        A network failure must not change the result silently: the affected
        Q-ids simply stay unresolved, and whatever is already in the cache is
        still used.
        """
        url = (
            "https://www.wikidata.org/w/api.php?action=wbgetentities"
            f"&ids={'|'.join(ids)}&props={props}&languages=en&format=json"
        )
        for attempt in range(3):
            try:
                request = Request(url, headers={"User-Agent": "dehergne-project/1.0 (research)"})
                with urlopen(request, timeout=30) as response:
                    return json.load(response).get("entities", {})
            except Exception as error:
                if attempt == 2:
                    print(f"  Wikidata lookup failed ({error}); leaving these Q-ids unresolved")
                    return {}
                time.sleep(2 * (attempt + 1))

    def claim_ids(claims, prop):
        return [
            claim["mainsnak"].get("datavalue", {}).get("value", {}).get("id")
            for claim in claims.get(prop, [])
        ]

    path = Path(cache_file)
    cache = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

    todo = [qid for qid in dict.fromkeys(qids) if qid and qid not in cache]
    for start in range(0, len(todo), 50):
        batch = todo[start : start + 50]
        entities = get(batch, "claims")
        country_of = {}
        for qid in batch:
            claims = entities.get(qid, {}).get("claims", {})
            countries = [c for c in claim_ids(claims, "P17") if c]
            if not countries:
                # no country statement: use the country of the entity it is located in
                parents = [p for p in claim_ids(claims, "P131") if p]
                if parents:
                    parent_claims = get(parents[:1], "claims").get(parents[0], {}).get("claims", {})
                    countries = [c for c in claim_ids(parent_claims, "P17") if c]
            country_of[qid] = countries[0] if countries else None
        country_ids = sorted({c for c in country_of.values() if c})
        labels = get(country_ids, "labels") if country_ids else {}
        for qid, country_id in country_of.items():
            label = None
            if country_id:
                label = (
                    labels.get(country_id, {}).get("labels", {}).get("en") or {}
                ).get("value")
            cache[qid] = {"country_id": country_id, "country_label": label}
        time.sleep(1)  # be polite to the Wikidata API

    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(cache, ensure_ascii=False, indent=1), encoding="utf-8")
    return {qid: cache.get(qid, {"country_id": None, "country_label": None}) for qid in qids}


if FETCH_MISSING_COUNTRIES and no_country:
    fetched = fetch_country_for_qids(no_country)
    for qid, record in fetched.items():
        print(f"  {qid}: {record['country_label']} ({record['country_id']})")
    # fill the gaps only: never overwrite a country already known
    for column in ("country_id", "country_label"):
        places[column] = places[column].fillna(
            places["wikidata_id"].map({q: r[column] for q, r in fetched.items()})
        )
    print(f"\nlocation attributes with a country now: "
          f"{places['country_label'].notna().sum()} / {len(places)}")
else:
    print("skipped: nothing missing, or FETCH_MISSING_COUNTRIES is False")

## Group historical states with the modern country

Wikidata's `P17` follows the **historical** state: a 17th-century Chinese
prefecture is *Ming dynasty* or *Qing dynasty*, an Indian one may be
*Portuguese India*. `COUNTRY_GROUPS` maps those raw labels to the country you
want to reason with. Labels that are not listed are kept as they are, so nothing
is lost.

In [ ]:
COUNTRY_GROUPS = {
    "China": [
        "People's Republic of China",
        "Qing dynasty",
        "Ming dynasty",
        "Republic of China",
        "Yuan dynasty",
    ],
    "India": ["India", "British Raj", "Mughal Empire", "Portuguese India"],
    "Portugal": ["Portugal", "Kingdom of Portugal"],
    "France": ["France", "Kingdom of France"],
    "Italy": ["Italy", "Papal States", "Kingdom of Italy", "Republic of Venice"],
    "Spain": ["Spain", "Crown of Castile", "Crown of Aragon"],
    "Japan": ["Japan", "Tokugawa shogunate", "Empire of Japan"],
    "Vietnam": ["Vietnam", "Đại Việt", "French Indochina"],
    "Germany": ["Germany", "Holy Roman Empire", "Kingdom of Prussia"],
}

# A few Q-ids used in the transcription are regions or historical states that
# have no P17 (*country*) statement on Wikidata (they show up as "(not
# identified)" in the report above). Assign by hand the ones you want counted
# under a country. Continents such as Asia or Europe are deliberately left out.
COUNTRY_BY_QID = {
    "Q29520": "China",  # "China" as a cultural region
    "Q83311": "Iran",   # Persian Empire
}

_COUNTRY_LOOKUP = {
    alias: group for group, aliases in COUNTRY_GROUPS.items() for alias in aliases
}


def country_group(label):
    """Country name to group by: the modern country, or the raw label."""
    if not isinstance(label, str) or not label.strip():
        return "(not identified)"
    return _COUNTRY_LOOKUP.get(label.strip(), label.strip())


places["country_label"] = places["country_label"].fillna(
    places["wikidata_id"].map(COUNTRY_BY_QID)
)
places["country"] = places["country_label"].apply(country_group)

# the rows the aggregations count: each person at their most specific place only
counted_places = places[places["most_specific"]] if KEEP_ONLY_MOST_SPECIFIC else places

# raw labels that are not in COUNTRY_GROUPS are used as they are
unmapped = sorted(set(places["country"]) - set(COUNTRY_GROUPS) - {"(not identified)"})
print(f"{len(unmapped)} raw country labels kept as-is "
      f"(add them to COUNTRY_GROUPS to merge them):")
unmapped

## The countries in the data

In [ ]:
country_totals = (
    counted_places.groupby("country")
    .agg(
        people=("person_id", "nunique"),
        places=("wikidata_id", "nunique"),
        events=("person_id", "size"),
        first_event=("place.date", "min"),
        last_event=("place.date", "max"),
    )
    .sort_values(["people", "events"], ascending=False)
)
country_totals

In [ ]:
import os

OUT_DIR = "../inferences/people-by-country"
os.makedirs(OUT_DIR, exist_ok=True)

country_totals.to_excel(f"{OUT_DIR}/country_totals.xlsx", sheet_name="countries")
print(f"written {OUT_DIR}/country_totals.xlsx")

## Choose a country

`country_rows` holds **every** location attribute of every person whose place is
in `COUNTRY`, with the `most_specific` flag. `country_df` is the subset the
aggregations count: when `KEEP_ONLY_MOST_SPECIFIC` is `True`, a person
documented both in Guangzhou and in Guangdong is counted in Guangzhou only.
Change the value and re-run the cells below to look at another country;
`COUNTRY_GROUPS` above controls which raw Wikidata labels belong to it.

In [ ]:
from timelink.kleio.utilities import format_timelink_date


def format_date(value):
    """Timelink date as text; '' when the attribute is undated (0)."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    try:
        return format_timelink_date(str(value))
    except Exception:
        return str(value)


COUNTRY = "China"

# every row of the selected country, with the date rendered for display
country_rows = places[places["country"] == COUNTRY].copy()
country_rows["date"] = country_rows["place.date"].apply(format_date)

# the rows the aggregations count: each person at their most specific place only
country_df = (
    country_rows[country_rows["most_specific"]].copy()
    if KEEP_ONLY_MOST_SPECIFIC
    else country_rows
)

print(f"{country_df['person_id'].nunique()} people, "
      f"{country_df['wikidata_id'].nunique()} distinct places, "
      f"{len(country_df)} location attributes in {COUNTRY}\n")
if KEEP_ONLY_MOST_SPECIFIC:
    print(f"{len(country_rows) - len(country_df)} of {len(country_rows)} attributes "
          f"are in a place that contains a more specific place of the same person\n")
country_df["place"].value_counts().head(20)

### Who was at each place

Grouped by the **Wikidata id** of the place, so that the many spellings of the
same place (`Macau`, `Macao`, `Macau (Colégio de S. Paulo)`) appear together.
Places without a Wikidata id are grouped by their transcribed name.

In [ ]:
def people_by_place(df, show_referenced=False):
    """Print the people documented at each place, in date order.

    Places are grouped by Wikidata id; places without one by name.

    ``show_referenced`` also lists the people merely *mentioned* in a biography
    (groupname 'referido'/'referida'), which repeats the missionaries under a
    second, occurrence-level id — off by default.
    """
    data = df if show_referenced else df[df["groupname"] == "n"]
    data = data.copy()
    data["place_key"] = data["wikidata_id"].fillna(
        "no-id: " + data["place"].astype(str)
    )

    for place_key, group in data.groupby("place_key"):
        variants = sorted(set(group["place"].astype(str)))
        modern = " / ".join(
            str(label)
            for label in (
                group["english_label"].dropna().unique()[:1].tolist()
                + group["chinese_label"].dropna().unique()[:1].tolist()
            )
            if label
        )
        if str(place_key).startswith("no-id: "):
            print(f"\n{place_key[len('no-id: '):]}   (no Wikidata id)")
        else:
            print(f"\n{modern or variants[0]}   [{place_key}]")
            print(f"    transcribed as: {' | '.join(variants)}")
        people = (
            group.sort_values(["place.date", "name"])
            .drop_duplicates(subset=["person_id", "place.type", "date"])
        )
        for _, row in people.iterrows():
            date = row["date"] if row["date"] else "-"
            print(f"    {date:<12}{row['name']} ({row['place.type']})  {row['person_id']}")


people_by_place(country_df)

### Flat table — one row per person and place

In [ ]:
COUNTRY_COLUMNS = [
    "person_id",
    "name",
    "groupname",
    "place",
    "place.type",
    "date",
    "place.original",
    "wikidata_id",
    "english_label",
    "chinese_label",
    "country",
    "country_label",
    "most_specific",
]

flat_table = (
    country_rows.sort_values(["place", "place.date", "name"])[COUNTRY_COLUMNS]
    .reset_index(drop=True)
)
flat_table.head(20)

### One row per person

`wikidata_ids` lists the places this person is documented in — the same
identifiers used by the place tables below, so people can be grouped by place
regardless of how the transcriber spelled it.

In [ ]:
def join_unique(values, separator=", "):
    """Sorted unique values as text, dropping missing ones."""
    return separator.join(sorted({str(value) for value in values if pd.notna(value)}))


people_summary = (
    country_df.groupby(["person_id", "name"])
    .agg(
        n_events=("place.type", "size"),
        n_places=("wikidata_id", "nunique"),
        wikidata_ids=("wikidata_id", join_unique),
        places=("place", join_unique),
        attribute_types=("place.type", join_unique),
        first_event=("place.date", "min"),
        last_event=("place.date", "max"),
    )
    .reset_index()
    .sort_values("name")
)
people_summary

### Top places

Every place of the selected country, ordered by number of people, with the
spellings that were merged into each one. Set `TOP_N` to show more or fewer;
the `places` sheet of the export below holds the complete list.

A place whose id is the country itself (e.g. `Q148`, used when a source says
only "China") is a country-level bucket, not a city.

In [ ]:
# Aggregation key for a place: the Wikidata id when there is one, else the
# transcribed name. Grouping on the id (and not on the place string) is what
# makes "Macau" / "Macao" / "Macau (Colégio)" one single place.
country_df["place_id"] = country_df["wikidata_id"].fillna(
    "no-id: " + country_df["place"].astype(str)
)

# One row per place, with all the people of that place aggregated.
place_totals = (
    country_df.groupby("place_id")
    .agg(
        wikidata_id=("wikidata_id", "first"),
        modern_name=("english_label", "first"),
        chinese_name=("chinese_label", "first"),
        transcribed_as=("place", lambda values: join_unique(values, " | ")),
        people=("person_id", "nunique"),
        person_ids=("person_id", join_unique),
        events=("person_id", "size"),
        first_event=("place.date", "min"),
        last_event=("place.date", "max"),
    )
    .sort_values(["people", "events"], ascending=False)
)
place_totals.index.name = "place_id"

# The same, one row per place and person (the people of each place listed).
place_people = (
    country_df.groupby(["place_id", "person_id", "name"])
    .agg(
        wikidata_id=("wikidata_id", "first"),
        n_events=("place.type", "size"),
        attribute_types=("place.type", join_unique),
        first_event=("place.date", "min"),
        last_event=("place.date", "max"),
    )
    .reset_index()
    .sort_values(["place_id", "name"])
)

# how many places to show in the table below
TOP_N = 20

place_totals.head(TOP_N)[
    [
        "wikidata_id",
        "modern_name",
        "chinese_name",
        "people",
        "events",
        "first_event",
        "last_event",
        "transcribed_as",
    ]
]

## Export the result for the selected country

One Excel file `../inferences/people-by-country/<country>.xlsx` with four sheets:

| sheet | one row per | contents |
|---|---|---|
| `places` | place (**Wikidata id**) | the aggregation of all the people in the same place: `people`, `events`, date range, and the transcribed spellings that map to it |
| `place_people` | place **and** person | the people of each place, listed by Wikidata id |
| `person_place` | person and place | the flat event table (one row per location attribute) |
| `people` | person | `n_places`, `wikidata_ids`, `places`, attribute types, date range |

Places are keyed by Wikidata id everywhere, so the many spellings of one place
(`Macau`, `Macao`, `Macau (Colégio de S. Paulo)`) are counted as one. Places
without a Wikidata id fall back to the transcribed name and are marked `no-id:`.

In [ ]:
import contextlib
import io
import re

slug = re.sub(r"[^A-Za-z0-9]+", "-", COUNTRY).strip("-").lower()
os.makedirs(OUT_DIR, exist_ok=True)
excel_file = f"{OUT_DIR}/{slug}.xlsx"

with pd.ExcelWriter(excel_file) as writer:
    place_totals.to_excel(writer, sheet_name="places")
    place_people.to_excel(writer, sheet_name="place_people", index=False)
    flat_table.to_excel(writer, sheet_name="person_place", index=False)
    people_summary.to_excel(writer, sheet_name="people", index=False)

# The TOP_N places only, with the people of each one, as a small workbook.
# NOTE: .xlsx, not .xls — .xls is the legacy binary format and the available
# engines (openpyxl / XlsxWriter) cannot write it; LibreOffice here converts
# to pdf/xlsx/ods/csv only. Renaming an .xlsx to .xls would mislabel it.
top_file = f"{OUT_DIR}/{slug}-top-{TOP_N}.xlsx"
top_places = place_totals.head(TOP_N)
rank_of = {place_id: rank for rank, place_id in enumerate(top_places.index, start=1)}
top_people = (
    place_people[place_people["place_id"].isin(top_places.index)]
    .assign(rank=lambda frame: frame["place_id"].map(rank_of))
    .merge(
        top_places[["modern_name", "chinese_name"]],
        left_on="place_id",
        right_index=True,
        how="left",
    )
    .sort_values(["rank", "name"])
)
top_people_columns = [
    "rank",
    "place_id",
    "wikidata_id",
    "modern_name",
    "chinese_name",
    "person_id",
    "name",
    "n_events",
    "attribute_types",
    "first_event",
    "last_event",
]
with pd.ExcelWriter(top_file) as writer:
    top_places.reset_index().to_excel(writer, sheet_name=f"top-{TOP_N}", index=False)
    top_people[top_people_columns].to_excel(writer, sheet_name="people", index=False)

# the same listing that was printed above, as a text file
buffer = io.StringIO()
with contextlib.redirect_stdout(buffer):
    people_by_place(country_df)
text_file = f"{OUT_DIR}/{slug}-people-by-place.txt"
with open(text_file, "w", encoding="utf-8") as handle:
    handle.write(buffer.getvalue())

print(f"{len(place_totals)} places, {len(place_people)} place/person rows, "
      f"{len(country_df)} events\n")
print(f"written {excel_file}")
print(f"written {top_file}")
print(f"written {text_file}")

for name in sorted(os.listdir(OUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUT_DIR, name)) / 1024
    print(f"  {name:<45}{size_kb:8.1f} kB")

## Drill down: one attribute

*"Who worked in China and entered the novitiate at Coimbra?"* The novitiate is
in Portugal and the mission in China, so this filters the **people** of the
selected country by an attribute recorded anywhere. `COUNTRY` still applies:
change `ATTRIBUTE` and `PLACE` and re-run. Set `PLACE = None` for every place of
that attribute, and `show_referenced=True` to include people only mentioned.

`PLACE` accepts a Wikidata id (`Q45412`), the transcribed name (`Coimbra`) or a
Wikidata label in English, Portuguese or Chinese (`Lisbon`, `Lisboa`, `里斯本`).

In [ ]:
import re


def matches_place(frame, place):
    """Rows whose place matches `place`.

    `place` is matched case-insensitively against
      * a Wikidata id, matched exactly            -- "Q45412"
      * the transcribed place name                -- "Coimbra", "Chinkiam, Tchen-kiang"
      * the Wikidata labels in English, Portuguese and Chinese
                                                  -- "Lisbon", "Lisboa", "澳門"

    Names are substrings, so "Coimbra" also matches "Coimbra (colégio)" and
    "Lisboa" also matches "Lisboa, Arroios". Pass the Q-id when you need
    exactly one place.
    """
    if place is None:
        return pd.Series(True, index=frame.index)
    wanted = str(place).strip()
    if re.fullmatch(r"[Qq]\d+", wanted):
        return frame["wikidata_id"].astype(str).str.upper() == wanted.upper()
    haystack = (
        frame["place"].astype(str)
        + " | " + frame["english_label"].astype(str)
        + " | " + frame["portuguese_label"].astype(str)
        + " | " + frame["chinese_label"].astype(str)
    ).str.casefold()
    return haystack.str.contains(wanted.casefold(), regex=False, na=False)


def people_with_attribute(attribute, place=None, country=COUNTRY, show_referenced=False):
    """People documented in `country` who have `attribute`, optionally at `place`.

    The attribute is searched over **all** places, not only those inside
    `country`: the novitiate at Coimbra is in Portugal while the mission is in
    China. Its "-x" variant is included, following the same rule as the counts.
    """
    wanted_types = {attribute, f"{attribute}-x"}
    selected = places[places["place.type"].isin(wanted_types)].copy()
    if country:
        people_of_country = set(country_rows["person_id"])
        selected = selected[selected["person_id"].isin(people_of_country)]
    if not show_referenced:
        selected = selected[selected["groupname"] == "n"]
    selected = selected[matches_place(selected, place)]
    selected["date"] = selected["place.date"].apply(format_date)
    return selected.sort_values(["name", "place.date"])


ATTRIBUTE = "jesuita-entrada"
PLACE = "Coimbra"

entries = people_with_attribute(ATTRIBUTE, PLACE)
print(f"{entries['person_id'].nunique()} people documented in {COUNTRY} "
      f"with {ATTRIBUTE} = {PLACE}\n")
entries[
    ["person_id", "name", "place", "date", "place.original", "wikidata_id", "english_label"]
]

In [ ]:
# The same query without the place filter: where the people of China entered the
# novitiate. Grouped by Wikidata id, so the spellings of one place ("Lisboa" and
# "Lisboa, Arroios") are counted together, as in the export above.
(
    people_with_attribute(ATTRIBUTE)
    .groupby("wikidata_id", dropna=False)
    .agg(
        places=("place", lambda values: " | ".join(sorted(set(values)))),
        people=("person_id", "nunique"),
        first=("date", "min"),
        last=("date", "max"),
    )
    .sort_values("people", ascending=False)
    .head(15)
)

### Their places in the selected country

For the people found above, every place they are documented at in `COUNTRY`,
exported to `../inferences/people-by-country/<country>_<attribute>_<place>.xlsx`
— here `china_jesuita-entrada_Coimbra.xlsx`, with two sheets:

| sheet | one row per | contents |
|---|---|---|
| `people` | person | `n_places`, the China places gathered in one cell, their Wikidata ids, attribute types and date range |
| `places` | person and place | the detail: place, attribute type, date, Wikidata id and labels |

`china_places` lists every place counted for that person in the country, not only
`estadia`; filter on `attribute_types` if you want stays only.

In [ ]:
import re


def slugify(text):
    """Filesystem-safe fragment for an export name."""
    return re.sub(r"[^A-Za-z0-9._-]+", "-", str(text)).strip("-")


# The people found by the drill-down above, and where they are documented in the
# selected country. `country_df` is the deduplicated frame, so these are the
# places the country aggregations count.
drill_people = set(entries["person_id"])
drill_places = (
    country_df[country_df["person_id"].isin(drill_people)]
    .sort_values(["person_id", "place.date", "place"])
)

# One row per person, with their places gathered in a single cell.
places_summary = (
    drill_places.groupby(["person_id", "name"])
    .agg(
        n_places=("place_id", "nunique"),
        n_events=("place.type", "size"),
        china_places=("place", join_unique),
        chinese_names=("chinese_label", join_unique),
        wikidata_ids=("wikidata_id", join_unique),
        attribute_types=("place.type", join_unique),
        first_event=("place.date", "min"),
        last_event=("place.date", "max"),
    )
    .reset_index()
    # keep the attribute that selected the person, so the file stands alone
    .merge(
        entries[["person_id", "place", "date"]].rename(
            columns={"place": f"{ATTRIBUTE}_place", "date": f"{ATTRIBUTE}_date"}
        ),
        on="person_id",
        how="left",
    )
    .sort_values("name")
)

drill_file = (
    f"{OUT_DIR}/{slugify(COUNTRY).lower()}_{slugify(ATTRIBUTE)}"
    f"_{slugify(PLACE) if PLACE else 'all'}.xlsx"
)
with pd.ExcelWriter(drill_file) as writer:
    places_summary.to_excel(writer, sheet_name="people", index=False)
    drill_places[
        [
            "person_id",
            "name",
            "place",
            "place.type",
            "date",
            "wikidata_id",
            "english_label",
            "chinese_label",
            "most_specific",
        ]
    ].to_excel(writer, sheet_name="places", index=False)

print(f"{len(places_summary)} people, {len(drill_places)} person/place rows")
print(f"written {drill_file}\n")
places_summary[["name", f"{ATTRIBUTE}_place", "n_places", "china_places"]].head(15)

## Notes on the data

* **Undated attributes.** `0` in `place.date` means the transcription has no
  date for that attribute (visible in the timeline as `-`). To borrow a date from
  the previous attribute of the same person, see the `ffill` step in
  [location-analysis-new](./location-analysis-new.ipynb).
* **`groupname`.** `n` is the missionary of the entry; `referido`/`referida` are
  people only mentioned in it, with occurrence-level ids (`...-ref1`). The listing
  above shows only `n`; use `people_by_place(country_df, show_referenced=True)`
  to include the others.
* **Merging linked occurrences.** A person transcribed in several entries may
  have more than one id. `effective_id()` and `normalize_persons()` in
  [dehergne_util.py](./dehergne_util.py) collapse them when a linked database is used.
* **Nested places (`most_specific`).** A person is counted once, at the deepest
  place documented for them, following the Wikidata `P131` chain. The rule only
  sees links that exist in the cache, and it cannot merge **two ids for the same
  place** — Qing Jiangsu (`Q28411809`) and modern Jiangsu (`Q16963`) are separate
  nodes, so nothing shadows either. Every row is kept in the `person_place`
  sheet with the flag, so the dropped rows can always be recovered.
* **No Wikidata id / no country.** Places that were not identified are invisible
  to this notebook. `locations_places_of_stay.xlsx` (from
  [location-analysis-new](./location-analysis-new.ipynb)) lists what is missing,
  and `wd_country_cache.json` records what the optional lookup found.
